# S4 · Проверка модели на новых сообщениях

Исторический реальный корпус UCI SMS Spam Collection, Almeida и Hidalgo, [DOI 10.24432/C5CC84](https://doi.org/10.24432/C5CC84), CC BY 4.0. Файл уже находится в `data/`, сеть не нужна.

Основной маршрут: 75 минут. Объект — одно сообщение. Повторы удаляются до разбиения. Словарь free, win, prize задан заранее. Параметры обучаются на обучающей части, порог выбирается на валидации. Блок 6 с тестом выполняется после фиксации решения. Условная цена заранее принята равной 5 × FP + FN.

Отправители и кампании неизвестны, похожие шаблоны могут оставаться. Этот опыт оценивает работу на отложенной части исторического корпуса.

### Подготовлено: импорты и чтение файла

In [ ]:
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

data_path = Path("data/sms-spam.tsv")
if not data_path.exists():
    data_path = Path("../data/sms-spam.tsv")
if not data_path.exists():
    data_path = Path("../../data/sms-spam.tsv")
raw_rows = []
for line in data_path.read_text(encoding="utf-8").splitlines():
    label, message = line.split("\t", 1)
    raw_rows.append((label, message))
print("Строк в исходном файле:", len(raw_rows))


## 1. Проверка данных и удаление повторов

`seen` хранит уже встретившиеся нормализованные тексты. `continue` переходит к следующей строке. После очистки: 5159 сообщений, из них 642 спам.

In [ ]:
# Блок 1. Проверка данных и удаление повторов


### Подготовлено: тот же признак, что на S3

In [ ]:
def count_words(message):
    count = 0
    for word in message.lower().split():
        word = word.strip(".,!?;:")
        if word in ["free", "win", "prize"]:
            count = count + 1
    return count


## 2. Таблица признаков и три части данных

В X одна строка содержит список из одного признака. `ids` — номера строк. `stratify` сохраняет близкие доли классов, `random_state` фиксирует разбиение. Размеры: 3095 / 1032 / 1032.

In [ ]:
# Блок 2. Таблица признаков и три части данных


## 3. Обучение библиотечной модели

`C=inf` отключает регуляризацию, чтобы сохранить функцию потерь из L1. Используется библиотечный оптимизатор L-BFGS. `max_iter` — предел его итераций. Две колонки predict_proba соответствуют classes_=[0,1]; [:,1] берёт вторую.

In [ ]:
# Блок 3. Обучение библиотечной модели


## 4. Два вида ошибок и точка отсчёта

Считаем случаи по одному. TN — обычное во входящих; FP — обычное в спаме; FN — спам во входящих; TP — спам в спаме. Сумма равна числу сообщений.

In [ ]:
# Блок 4. Два вида ошибок и точка отсчёта


## 5. Выбор порога на валидации

Выполняем только четыре заранее заданных сравнения на валидации. При одинаковой цене сохраняется первый порог. Фиксируем 0.75.

In [ ]:
# Блок 5. Выбор порога на валидации


### Подготовлено: реальные ошибки только на валидации

In [ ]:
for error_name, true_label, predicted_label in [("FP", 0, 1), ("FN", 1, 0)]:
    for i in range(len(validation_ids)):
        row_id = validation_ids[i]
        prediction = int(validation_probability[i] >= best_threshold)
        if y[row_id] == true_label and prediction == predicted_label:
            print(error_name, "x =", X[row_id, 0], "p =", round(validation_probability[i], 3))
            print(messages[row_id])
            break


## 6. Единственная итоговая проверка

Все решения приняты до этого блока. На этой версии данных знаменатели precision и recall положительны. В общем случае при нулевом знаменателе доля не определена и требует явной договорённости. Результат: 9 верных срабатываний из 9, найдено 9 из 128 сообщений спама. Изменения после просмотра теста требуют новой независимой проверки.

In [ ]:
# Блок 6. Единственная итоговая проверка
